# 6. Clustering and resolution selection

After dimensionality reduction, the next step is to define transcriptionally coherent cell clusters. Clustering provides a discrete partition of the embedding that is later used for annotation, differential expression, and downstream analyses such as compositional shifts.

In this notebook, we:

- construct a k-nearest-neighbor (KNN) graph on the PCA embedding,
- compute a UMAP embedding for visualization,
- run Leiden clustering at multiple resolutions, and
- quantify clustering quality at each resolution using the silhouette score.

The output is an AnnData object with multiple Leiden clusterings stored, along with UMAP coordinates and silhouette-based guidance for choosing an appropriate resolution.

## 6.1. Imports and Scanpy settings

In [ ]:
import numpy as np
import scanpy as sc
from sklearn.metrics import silhouette_score

# Scanpy verbosity and plotting
sc.settings.verbosity = 0
sc.settings.set_figure_params(dpi=80, facecolor="white", frameon=False)

## 6.2. Load dimensionality-reduced AnnData

We load the AnnData object that already contains:

- a PCA embedding in `.obsm["X_pca"]`, and
- a UMAP/t-SNE embedding from the previous notebook (if desired).

This object will serve as the basis for graph construction and clustering.

In [ ]:
input_file = "dimensionality_reduced.h5ad"
print(f"Reading input AnnData: {input_file}")

adata = sc.read(input_file)
adata

## 6.3. Construct KNN graph

We construct a k-nearest-neighbor graph on the PCA-reduced space. This graph represents local neighborhood relationships between cells and is the basis for Leiden clustering and UMAP.

In [ ]:
print("Constructing KNN graph on PCA space...")
sc.pp.neighbors(adata, n_pcs=30)

## 6.4. Compute UMAP embedding

We compute a UMAP embedding from the neighbor graph for visualization. This allows us to visually assess the structure of the data and how clusters separate in a low-dimensional space.

In [ ]:
print("Computing UMAP embedding...")
sc.tl.umap(adata)

## 6.5. Leiden clustering at multiple resolutions

To explore different levels of granularity, we run Leiden clustering at several resolutions. Lower resolutions produce fewer, larger clusters; higher resolutions produce more, finer clusters.

In [ ]:
print("Running Leiden clustering at multiple resolutions...")

sc.tl.leiden(adata, key_added="leiden_res0_25", resolution=0.25)
sc.tl.leiden(adata, key_added="leiden_res0_5", resolution=0.5)
sc.tl.leiden(adata, key_added="leiden_res1",   resolution=1.0)
sc.tl.leiden(adata, key_added="leiden_res2",   resolution=2.0)

## 6.6. Visualize clustering on UMAP

We overlay the different Leiden clusterings on the UMAP embedding to visually compare how resolution affects the number and separation of clusters.

In [ ]:
print("Visualizing clustering results on UMAP...")
sc.pl.umap(
    adata,
    color=["leiden_res0_25", "leiden_res0_5", "leiden_res1", "leiden_res2"],
    legend_loc="on data",
)

## 6.7. Silhouette score evaluation

To quantitatively assess cluster compactness and separation, we compute the silhouette score for each Leiden clustering using the PCA representation. Higher scores indicate more coherent and well-separated clusters.

In [ ]:
res_keys = ["leiden_res0_25", "leiden_res0_5", "leiden_res1", "leiden_res2"]

X = adata.obsm["X_pca"]

for key in res_keys:
    labels = adata.obs[key].astype(int)
    n_clusters = len(np.unique(labels))

    # Need >1 cluster & <N clusters for silhouette to be meaningful
    if 1 < n_clusters < adata.n_obs:
        sil_score = silhouette_score(X, labels)
        print(f"Silhouette score for {key}: {sil_score:.3f} (n_clusters = {n_clusters})")
    else:
        print(f"{key} not valid for silhouette analysis (all same or all unique labels).")

## 6.8. Save clustered AnnData and next steps

We now save the AnnData object with:

- KNN graph,
- UMAP embedding, and
- multiple Leiden clusterings,

as well as the silhouette scores printed above. This clustered object will be used for downstream steps such as cluster annotation, GABAergic reclustering, and compositional analyses.

In [ ]:
output_file = "clustered.h5ad"
adata.write(output_file)

print("Processing completed.")
print(f"Saved clustered AnnData to: {output_file}")